In [1]:


import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report,
                              confusion_matrix)

RANDOM_STATE = 42

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
})

SEASON_ORDER = ["Season A (Rainy)", "Season B (Rainy)", "Dry Season"]
SEASON_COLORS = {"Season A (Rainy)": "#2E86AB",
                  "Season B (Rainy)": "#4CAF91",
                  "Dry Season": "#D9A441"}

# --------------------------------------------------------------------------
# 1. LOAD DATA AND BUILD FEATURE SET
# --------------------------------------------------------------------------
print("=" * 150)
print("STEP 1: DATA LOADING & BUILDING FEATURE SET")
print("=" * 150)

df = pd.read_csv("Processed Data.csv")

district_cols = [c for c in df.columns if c.startswith("District_")]
climate_cols = ["tas", "tasmin", "tasmax", "temp_range",
                 "pr", "pr_lag1", "pr_lag2", "pr_lag3"]
FEATURE_COLS = climate_cols + district_cols

X = df[FEATURE_COLS].copy()
y = df["Season"].copy()

print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Target: Season -- classes: {sorted(y.unique())}")
print("\nNOTE: 'Month' and 'Year' deliberately left out from features")
print("      to stop the model from memorizing the calendar")
print("      so it can learn the underlying climate patterns.")

# --------------------------------------------------------------------------
# 2. TRAIN / TEST SPLIT
# --------------------------------------------------------------------------
print("\n" + "=" * 150)
print("STEP 2: TRAIN / TEST SPLIT")
print("=" * 150)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

split_rows = []
for label, series in [("Train", y_train), ("Test", y_test)]:
    for s_ in SEASON_ORDER:
        n = (series == s_).sum()
        split_rows.append({
            "Split": label, "Season": s_, "N": int(n),
            "Percent of Split (%)": round(n / len(series) * 100, 1)
        })
split_summary = pd.DataFrame(split_rows)
print(split_summary.to_string(index=False))
split_summary.to_csv("Train_Test Split Summary.csv", index=False)

print(f"\nTotal training observations: {len(X_train)}")
print(f"Total test observations    : {len(X_test)}")

# --------------------------------------------------------------------------
# 3. FEATURE SCALING
# --------------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------------------------------------
# 4. TRAIN THE SELECTED MODELS
# --------------------------------------------------------------------------
print("\n" + "=" * 150)
print("STEP 3: MODEL TRAINING")
print("=" * 150)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "SVM (RBF Kernel)": SVC(
        kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE
    ),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions[name] = model.predict(X_test_scaled)
    print(f"-> {name} trained successfully.")

# --------------------------------------------------------------------------
# 5. MODEL PERFORMANCE COMPARISON
# --------------------------------------------------------------------------
print("\n" + "=" * 150)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 150)

perf_rows = []
for name, y_pred in predictions.items():
    perf_rows.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision (macro)": round(precision_score(y_test, y_pred, average="macro"), 4),
        "Recall (macro)": round(recall_score(y_test, y_pred, average="macro"), 4),
        "F1-Score (macro)": round(f1_score(y_test, y_pred, average="macro"), 4),
        "F1-Score (weighted)": round(f1_score(y_test, y_pred, average="weighted"), 4),
    })
performance_summary = pd.DataFrame(perf_rows).sort_values(
    "F1-Score (macro)", ascending=False
)
print(performance_summary.to_string(index=False))
performance_summary.to_csv("Model Performance Comparison.csv", index=False)

# --------------------------------------------------------------------------
# 6. PER-CLASS CLASSIFICATION REPORT
# --------------------------------------------------------------------------
print("\n" + "=" * 150)
print("PER-CLASS CLASSIFICATION")
print("=" * 150)

report_rows = []
for name, y_pred in predictions.items():
    report = classification_report(y_test, y_pred, output_dict=True)
    for season in SEASON_ORDER:
        r = report[season]
        report_rows.append({
            "Model": name, "Season": season,
            "Precision": round(r["precision"], 3),
            "Recall": round(r["recall"], 3),
            "F1-Score": round(r["f1-score"], 3),
            "Support": int(r["support"]),
        })
classification_long = pd.DataFrame(report_rows)
print(classification_long.to_string(index=False))
classification_long.to_csv("Classification Report.csv", index=False)

# --------------------------------------------------------------------------
# 7. CONFUSION MATRICES 
# --------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
short_labels = ["Season A (Rainy)", "Season B (Rainy)", "Dry Season"]
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=SEASON_ORDER)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short_labels, yticklabels=short_labels,
                cbar=False, ax=ax, square=True)
    ax.set_title(name)
    ax.set_xlabel("Predicted Season")
    ax.set_ylabel("Actual Season")
fig.suptitle("Confusion Matrices: Season Classification by Model",
             fontweight="normal", fontsize=14)
plt.tight_layout()
plt.savefig("Confusion Matrices.png")
plt.close()
print("\nSaved: Confusion Matrices.png")

# --------------------------------------------------------------------------
# 8.MODEL PERFORMANCE COMPARISON 
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 5.5))
metrics_to_plot = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-Score (macro)"]
plot_df = performance_summary.set_index("Model")[metrics_to_plot]
plot_df.plot(kind="bar", ax=ax, color=["#2E86AB", "#4CAF91", "#D9A441", "#D64545"])
ax.set_title("Model Performance Comparison — Seasonal Classification")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right", fontsize=9.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Performance Comparison.png")
plt.close()
print("Saved: Performance Comparison.png")

# --------------------------------------------------------------------------
# 9. RANDOM FOREST FEATURE IMPORTANCE
# --------------------------------------------------------------------------
print("\n" + "=" * 150)
print("FEATURE IMPORTANCE ON RANDOM FOREST")
print("=" * 150)

rf_model = models["Random Forest"]
importance_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)
print(importance_df.to_string(index=False))
importance_df.to_csv("feature importance.csv", index=False)

fig, ax = plt.subplots(figsize=(9.5, 6.5))
sns.barplot(data=importance_df, x="Importance", y="Feature",
            color="#2E86AB", ax=ax)
ax.set_title("Random Forest Feature Importance — Season Classification")
ax.set_xlabel("Relative Importance")
plt.tight_layout()
plt.savefig("Feature Importance.png")
plt.close()
print("\nSaved: Feature Importance.png")

# --------------------------------------------------------------------------
# 10. PCA VISUALIZATION OF SEASONAL SEPARATION
# --------------------------------------------------------------------------
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(StandardScaler().fit_transform(X))
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Season"] = y.values

fig, ax = plt.subplots(figsize=(8.5, 6.5))
for s_ in SEASON_ORDER:
    subset = pca_df[pca_df["Season"] == s_]
    ax.scatter(subset["PC1"], subset["PC2"], s=8, alpha=0.35,
               color=SEASON_COLORS[s_], label=s_)
ax.set_title("Principal Component Analysis Projection of Climate Features, by Season")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig("PCA Separation of Seasons.png")
plt.close()
print("Saved: PCA Separation of Seasons.png")

print("\n" + "=" * 150)
print("SEASONAL PATTERN ANALYSIS SUCCESSFULLY COMPLETED")
print(f"Best model by macro F1-score: {performance_summary.iloc[0]['Model']} "
      f"({performance_summary.iloc[0]['F1-Score (macro)']})")
print("=" * 150)


STEP 1: DATA LOADING & BUILDING FEATURE SET
Feature columns (15): ['tas', 'tasmin', 'tasmax', 'temp_range', 'pr', 'pr_lag1', 'pr_lag2', 'pr_lag3', 'District_Burera', 'District_Gakenke', 'District_Musanze', 'District_Ngororero', 'District_Nyabihu', 'District_Rubavu', 'District_Rutsiro']
Target: Season -- classes: ['Dry Season', 'Season A (Rainy)', 'Season B (Rainy)']

NOTE: 'Month' and 'Year' deliberately left out from features
      to stop the model from memorizing the calendar
      so it can learn the underlying climate patterns.

STEP 2: TRAIN / TEST SPLIT
Split           Season    N  Percent of Split (%)
Train Season A (Rainy) 4992                  33.3
Train Season B (Rainy) 3744                  25.0
Train       Dry Season 6240                  41.7
 Test Season A (Rainy) 1248                  33.3
 Test Season B (Rainy)  936                  25.0
 Test       Dry Season 1560                  41.7

Total training observations: 14976
Total test observations    : 3744

STEP 3: MODE